# TransLOB Baseline Notebook
This notebook implements and trains the baseline encoder model under our unified, leakage-safe LOBench replication pipeline.

In [1]:
# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

Mounted at /content/drive
Mounted Google Drive and changed directory to baselines.


In [2]:
# Install PyTorch Lightning if it is not present in the environment
!pip install -q lightning pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 59.5 MB/s eta 0:00:00


In [3]:
from common import *
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Libraries and common module imported successfully.')

Libraries and common module imported successfully.


In [4]:
class CausalConv1d(nn.Module):
    """1D convolution padded so output[t] only depends on input[<=t] -- causal."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=self.pad, dilation=dilation)

    def forward(self, x):  # x: [B, C, T]
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad > 0 else out

class TransLOBEncoder(nn.Module):
    def __init__(self, n_features=40, conv_channels=32, d_model=64, nhead=4,
                 num_transformer_layers=2, latent_dim=256, seq_len=100):
        super().__init__()
        # Dilated causal convolution stack, dilations 1,2,4,8 (standard WaveNet-style schedule)
        self.conv_layers = nn.ModuleList([
            CausalConv1d(n_features if i == 0 else conv_channels, conv_channels,
                         kernel_size=3, dilation=2**i)
            for i in range(4)
        ])
        self.conv_proj = nn.Linear(conv_channels, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        self.proj = nn.Linear(d_model, latent_dim)

    def forward(self, x):  # x: [B, 100, 40]
        h = x.transpose(1, 2)                     # [B, 40, 100]
        for conv in self.conv_layers:
            h = F.relu(conv(h))
        h = h.transpose(1, 2)                       # [B, 100, conv_channels]
        h = self.conv_proj(h) + self.pos_embedding   # [B, 100, d_model]
        # Causal mask: position t may only attend to positions <= t
        seq_len_dim = h.shape[1]
        mask = torch.triu(torch.full((seq_len_dim, seq_len_dim), float('-inf')), diagonal=1).to(h.device)
        h = self.transformer(h, mask=mask)
        last = h[:, -1, :]                            # [B, d_model]
        return self.proj(last)                         # [B, latent_dim]

In [5]:
model_name = 'TransLOB'
stocks = ['sz000001', 'sz000002', 'sz000858', 'sz300147', 'sz002415']
for stock in stocks:
    print(f'\n========================================')
    print(f'Starting experiment for Model: {model_name} | Stock: {stock}')
    print(f'========================================')
    run_experiment(
        encoder_class=TransLOBEncoder,
        model_name=model_name,
        stock=stock,
        latent_dim=256,
        max_epochs=100
    )


Starting experiment for Model: TransLOB | Stock: sz000001
Loading data from data/sz000001-level10_processed.csv...
Loaded shape: (1171534, 41)
Columns and Level 10 ordering verified successfully.
Detecting sessions (3-second continuous chunks)...
Detected 488 sessions.
Splitting train/val/test splits (80/10/10 by calendar dates)...
Constructing sequence window indices (seq_len=100, stride=1)...

--- Train/Val/Test Splits & Windows ---
Split      | Rows       | Window Count
Train      | 936254     | 897644
Validation | 115231     | 110479
Test       | 120049     | 115099
---------------------------------------

Encoder parameters: 138,304
Shared Decoder parameters: 4,756,896
Total model parameters: 4,895,200


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Found existing checkpoint at /content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/TransLOB/sz000001/last.ckpt. Resuming training...


/usr/local/lib/python3.12/dist-packages/lightning/fabric/loggers/csv_logs.py:268: Experiment logs directory training_logs/TransLOB/sz000001 exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/TransLOB/sz000001 exists and is not empty.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/TransLOB/sz000001/last.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/TransLOB/sz000001/last.ckpt
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:566: The dirpath has changed from '/content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/checkpoints/TransLOB/sz0

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ TransLOBEncoder │  138 K │ train │     0 │
│ 1 │ decoder │ SharedDecoder   │  4.8 M │ train │     0 │
│ 2 │ loss_fn │ MSELoss         │      0 │ train │     0 │
│ 3 │ mae_fn  │ L1Loss          │      0 │ train │     0 │
└───┴─────────┴─────────────────┴────────┴───────┴───────┘

...

Training loop finished in 11062 seconds.
Loading best checkpoint for evaluation: /content/drive/MyDrive/JEPA_LOB/baselines/checkpoints/TransLOB/sz002415/best-v1.ckpt


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mae          │    0.12630873918533325    │
│         test_mse          │    0.09068084508180618    │
└───────────────────────────┴───────────────────────────┘


=== FINAL REPORT ===
Model: TransLOB | Stock: sz002415
Encoder params: 138,304
Total params (encoder + shared decoder): 4,895,200
Training time: 11062s
Test MSE: 0.0907
Test MAE: 0.1263

